# Anomaly Detection Experiments — Original Features (Stage 1 and Stage 2)

This notebook contains leakage-controlled anomaly and novelty experiments using Isolation Forest, Local Outlier Factor (novelty mode), and One-Class SVM on the 41 original features.

1. Global Normal-vs-Attack detection with runs named `original_{Algorithm}`.
2. Normal-vs-attack-family specialists with runs named `Original_Normalvs{Attack_category}_{Algorithm}`.
3. **Stage-2 global attack novelty** using leave-one-attack-family-out pseudo-unknown evaluation, with runs named `Original_global_attack_{Algorithm}`.
4. **Stage-2 attack-category novelty** using one model per family, with runs named `Original_{Attack_category}_{Algorithm}`.

The Stage-2 experiments receive attack traffic only. Because the dataset has no explicit unknown-attack class, global novelty is evaluated by withholding each known attack family in turn and treating it as pseudo-unknown. Category novelty treats the selected family as the known inlier class and the other attack families as out-of-category traffic. Validation/reference data selects thresholds; the held-out test split is used only for final evaluation.

In [1]:
import json, platform, sys, time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow, mlflow.pyfunc, mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, accuracy_score, average_precision_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, precision_recall_fscore_support, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.svm import OneClassSVM
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from configs.config import EXPERIMENT_NAME, RANDOM_STATE
TRACKING_DB = (PROJECT_ROOT / 'Notebooks' / 'mlflow.db').resolve()
mlflow.set_tracking_uri(f'sqlite:///{TRACKING_DB.as_posix()}')
mlflow.set_experiment(EXPERIMENT_NAME)
NORMAL_TRAIN_LIMIT = 6000
NORMAL_VALIDATION_LIMIT = 6000
ATTACK_CATEGORIES = ['DoS', 'Probe', 'R2L', 'U2R']
ALGORITHMS = ['IsolationForest', 'LocalOutlierFactor', 'OneClassSVM']
print('MLflow:', mlflow.get_tracking_uri(), '| experiment:', EXPERIMENT_NAME)

MLflow: sqlite:///D:/E Drive/Sentiflow-Network Intrusion Detection/Notebooks/mlflow.db | experiment: Network_Intrusion_Detection


In [2]:
data_path = PROJECT_ROOT / 'Data' / 'Consolidated_df.csv'
df = pd.read_csv(data_path)
ORIGINAL_FEATURES = ('duration','protocoltype','service','flag','srcbytes','dstbytes','land','wrongfragment','urgent','hot','numfailedlogins','loggedin','numcompromised','rootshell','suattempted','numroot','numfilecreations','numshells','numaccessfiles','numoutboundcmds','ishostlogin','isguestlogin','count','srvcount','serrorrate','srvserrorrate','rerrorrate','srvrerrorrate','samesrvrate','diffsrvrate','srvdiffhostrate','dsthostcount','dsthostsrvcount','dsthostsamesrvrate','dsthostdiffsrvrate','dsthostsamesrcportrate','dsthostsrvdiffhostrate','dsthostserrorrate','dsthostsrvserrorrate','dsthostrerrorrate','dsthostsrvrerrorrate')
X = df[list(ORIGINAL_FEATURES)].copy()
families = df['attack_category'].astype(str)
train_idx, holdout_idx = train_test_split(np.arange(len(df)), test_size=0.40, stratify=families, random_state=RANDOM_STATE)
val_idx, test_idx = train_test_split(holdout_idx, test_size=0.50, stratify=families.iloc[holdout_idx], random_state=RANDOM_STATE + 1)
normal_train_idx = train_idx[families.iloc[train_idx].to_numpy() == 'Normal']
if len(normal_train_idx) > NORMAL_TRAIN_LIMIT:
    normal_train_idx, _ = train_test_split(normal_train_idx, train_size=NORMAL_TRAIN_LIMIT, random_state=RANDOM_STATE)
normal_val_idx = val_idx[families.iloc[val_idx].to_numpy() == 'Normal']
if len(normal_val_idx) > NORMAL_VALIDATION_LIMIT:
    normal_val_idx, _ = train_test_split(normal_val_idx, train_size=NORMAL_VALIDATION_LIMIT, random_state=RANDOM_STATE)
categorical_features = X.select_dtypes(include=['object','category','string']).columns.tolist()
numeric_features = [c for c in ORIGINAL_FEATURES if c not in categorical_features]
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
], remainder='drop', verbose_feature_names_out=False)
X_normal_train = preprocessor.fit_transform(X.iloc[normal_train_idx])
X_val_all = preprocessor.transform(X.iloc[val_idx])
X_test_all = preprocessor.transform(X.iloc[test_idx])
family_val = families.iloc[val_idx].to_numpy(); family_test = families.iloc[test_idx].to_numpy()
split_summary = pd.DataFrame({'train': families.iloc[train_idx].value_counts(), 'validation': families.iloc[val_idx].value_counts(), 'test': families.iloc[test_idx].value_counts()}).fillna(0).astype(int)
print(f'Original features={len(ORIGINAL_FEATURES)}, encoded dimensions={X_normal_train.shape[1]}, normal fit rows={len(normal_train_idx)}')
display(split_summary)

Original features=41, encoded dimensions=71, normal fit rows=6000


,train,validation,test
attack_category,,,
Normal,40405,13469,13469
DoS,27556,9185,9186
Probe,6994,2331,2331
R2L,597,199,199
U2R,31,11,10


In [3]:
DEFAULT_PARAMS = {
 'IsolationForest': {'n_estimators': 300, 'max_samples': 1.0, 'contamination': 'auto', 'n_jobs': -1},
 'LocalOutlierFactor': {'n_neighbors': 35, 'contamination': 0.05, 'novelty': True, 'n_jobs': -1},
 'OneClassSVM': {'kernel': 'rbf', 'nu': 0.05, 'gamma': 'scale'},
}
def build_detector(name, params=None):
    p = {**DEFAULT_PARAMS[name], **(params or {})}
    if name == 'IsolationForest': return IsolationForest(random_state=RANDOM_STATE, **p)
    if name == 'LocalOutlierFactor': return LocalOutlierFactor(**p)
    return OneClassSVM(**p)
def anomaly_score(model, values): return -np.asarray(model.decision_function(values)).ravel()
def binary_metrics(y_true, prediction, scores):
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0,1]).ravel()
    return {'accuracy': accuracy_score(y_true,prediction), 'balanced_accuracy': balanced_accuracy_score(y_true,prediction), 'precision': precision_score(y_true,prediction,zero_division=0), 'recall': recall_score(y_true,prediction,zero_division=0), 'f1': f1_score(y_true,prediction,zero_division=0), 'roc_auc': roc_auc_score(y_true,scores), 'average_precision': average_precision_score(y_true,scores), 'false_positive_rate': fp/max(fp+tn,1), 'detection_rate': tp/max(tp+fn,1), 'specificity': tn/max(tn+fp,1), 'true_negatives': int(tn), 'false_positives': int(fp), 'false_negatives': int(fn), 'true_positives': int(tp)}
def binary_figure(y_true, prediction, scores, threshold, title):
    fig, axes = plt.subplots(2,2,figsize=(11,8))
    ConfusionMatrixDisplay.from_predictions(y_true,prediction,display_labels=['Normal','Attack'],cmap='Blues',ax=axes[0,0],colorbar=False)
    RocCurveDisplay.from_predictions(y_true,scores,ax=axes[0,1]); axes[0,1].set_title('ROC curve')
    PrecisionRecallDisplay.from_predictions(y_true,scores,ax=axes[1,0]); axes[1,0].set_title('Precision–Recall curve')
    axes[1,1].hist(scores[y_true==0],bins=40,alpha=.6,label='Normal'); axes[1,1].hist(scores[y_true==1],bins=40,alpha=.6,label='Attack'); axes[1,1].axvline(threshold,color='black',ls='--',label='threshold'); axes[1,1].legend(); axes[1,1].set_title('Anomaly-score distribution')
    fig.suptitle(title); fig.tight_layout(); return fig
def tracking_dataset(indices, target_name, name):
    frame=X.iloc[indices].copy(); frame[target_name]=families.iloc[indices].to_numpy()
    return mlflow.data.from_pandas(frame,source=str(data_path.resolve()),targets=target_name,name=name)
normal_training_dataset = tracking_dataset(normal_train_idx,'attack_category','normal_only_training_original')
common_metadata = {'original_features': list(ORIGINAL_FEATURES), 'categorical_features': categorical_features, 'numeric_features': numeric_features, 'normal_only_training': True, 'split_strategy': '60/20/20 stratified by attack_category', 'python_version': platform.python_version(), 'sklearn_version': sklearn.__version__, 'input_schema': {c: str(X[c].dtype) for c in ORIGINAL_FEATURES}}

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


## Experiment 1 — Global Normal vs Attack anomaly detection

The detection threshold is the 95th percentile of each detector's anomaly score on normal-only training data. The test set is untouched until final evaluation.

In [4]:
global_models={}; global_results=[]
y_test_binary=(family_test!='Normal').astype(int)
global_test_dataset=tracking_dataset(test_idx,'attack_category','global_anomaly_test_original')
for algorithm in ALGORITHMS:
    started=time.perf_counter(); model=build_detector(algorithm); model.fit(X_normal_train); fit_seconds=time.perf_counter()-started
    train_scores=anomaly_score(model,X_normal_train); threshold=float(np.quantile(train_scores,0.95))
    scores=anomaly_score(model,X_test_all); prediction=(scores>=threshold).astype(int); metrics=binary_metrics(y_test_binary,prediction,scores); metrics['fit_seconds']=fit_seconds
    run_name=f'original_{algorithm}'
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({'task':'global_anomaly_detection','data_variant':'Original','algorithm':algorithm,'training_labels_used':'normal_filter_only'})
        mlflow.log_input(normal_training_dataset,context='normal_only_training'); mlflow.log_input(global_test_dataset,context='held_out_evaluation')
        mlflow.log_params({**DEFAULT_PARAMS[algorithm],'algorithm':algorithm,'threshold':threshold,'threshold_quantile':0.95,'original_feature_count':len(ORIGINAL_FEATURES),'encoded_dimensions':X_normal_train.shape[1],'normal_train_rows':len(normal_train_idx),'test_rows':len(test_idx),'random_state':RANDOM_STATE})
        mlflow.log_metrics({k:float(v) for k,v in metrics.items()})
        fig=binary_figure(y_test_binary,prediction,scores,threshold,run_name); mlflow.log_figure(fig,'plots/global_binary_diagnostics.png'); plt.close(fig)
        mlflow.log_dict(common_metadata,'metadata/run_metadata.json'); mlflow.log_dict({'score_orientation':'larger means more anomalous','decision_rule':'anomaly_score >= threshold','output_labels':{'0':'Normal','1':'Attack'}},'metadata/input_output_contract.json')
        mlflow.sklearn.log_model(sk_model=model,name='anomaly_detector')
        global_results.append({'run_name':run_name,'run_id':run.info.run_id,'algorithm':algorithm,'threshold':threshold,**metrics})
    global_models[algorithm]={'model':model,'threshold':threshold}
global_results_df=pd.DataFrame(global_results).sort_values('f1',ascending=False).reset_index(drop=True)
display(global_results_df[['run_name','accuracy','balanced_accuracy','precision','recall','f1','roc_auc','average_precision','false_positive_rate','detection_rate']])

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,run_name,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,false_positive_rate,detection_rate
0,original_IsolationForest,0.947847,0.948015,0.938289,0.950452,0.944331,0.987736,0.986183,0.054421,0.950452
1,original_OneClassSVM,0.935384,0.934499,0.938357,0.921712,0.929960,0.969985,0.965690,0.052714,0.921712
2,original_LocalOutlierFactor,0.625481,0.601575,0.808293,0.256012,0.388860,0.890752,0.796673,0.052862,0.256012


## Experiment 2 — Attack-wise binary specialists

Each global normal-only detector is evaluated separately on Normal vs one attack family. This isolates family detection rate and identifies algorithm specialization without retraining on test labels.

In [5]:
attack_results=[]
for category in ATTACK_CATEGORIES:
    mask=(family_test=='Normal')|(family_test==category); category_indices=test_idx[mask]; values=X_test_all[mask]; y_category=(family_test[mask]==category).astype(int)
    category_dataset=tracking_dataset(category_indices,'attack_category',f'normal_vs_{category}_test_original')
    for algorithm in ALGORITHMS:
        model=global_models[algorithm]['model']; threshold=global_models[algorithm]['threshold']; scores=anomaly_score(model,values); prediction=(scores>=threshold).astype(int); metrics=binary_metrics(y_category,prediction,scores)
        run_name=f'Original_Normalvs{category}_{algorithm}'
        with mlflow.start_run(run_name=run_name) as run:
            mlflow.set_tags({'task':'attack_wise_anomaly_detection','data_variant':'Original','attack_category':category,'algorithm':algorithm,'training_labels_used':'normal_filter_only'})
            mlflow.log_input(normal_training_dataset,context='normal_only_training'); mlflow.log_input(category_dataset,context='held_out_attack_wise_evaluation')
            mlflow.log_params({**DEFAULT_PARAMS[algorithm],'algorithm':algorithm,'attack_category':category,'threshold':threshold,'threshold_source':'normal_training_95th_percentile','original_feature_count':len(ORIGINAL_FEATURES),'encoded_dimensions':X_normal_train.shape[1],'normal_train_rows':len(normal_train_idx),'normal_test_rows':int((y_category==0).sum()),'attack_test_rows':int((y_category==1).sum()),'random_state':RANDOM_STATE})
            mlflow.log_metrics({k:float(v) for k,v in metrics.items()})
            fig=binary_figure(y_category,prediction,scores,threshold,run_name); mlflow.log_figure(fig,f'plots/{category}_binary_diagnostics.png'); plt.close(fig)
            mlflow.log_dict({**common_metadata,'evaluated_attack_category':category,'class_counts':{'Normal':int((y_category==0).sum()),category:int((y_category==1).sum())}},'metadata/run_metadata.json')
            mlflow.sklearn.log_model(sk_model=model,name='anomaly_detector')
            attack_results.append({'run_name':run_name,'run_id':run.info.run_id,'attack_category':category,'algorithm':algorithm,'threshold':threshold,'support':int(y_category.sum()),**metrics})
attack_results_df=pd.DataFrame(attack_results)
best_base_by_attack=attack_results_df.sort_values(['f1','detection_rate'],ascending=False).groupby('attack_category',as_index=False).first()
display(Markdown('### All attack-wise results')); display(attack_results_df[['attack_category','algorithm','support','false_positive_rate','detection_rate','precision','f1','roc_auc']].sort_values(['attack_category','f1'],ascending=[True,False]))
display(Markdown('### Best untuned detector per attack family')); display(best_base_by_attack[['attack_category','algorithm','support','false_positive_rate','detection_rate','precision','f1']])

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


...

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


### All attack-wise results

,attack_category,algorithm,support,false_positive_rate,detection_rate,precision,f1,roc_auc
0,DoS,IsolationForest,9186,0.054421,0.969301,0.923939,0.946077,0.990975
2,DoS,OneClassSVM,9186,0.052714,0.949815,0.924748,0.937114,0.976483
1,DoS,LocalOutlierFactor,9186,0.052862,0.216852,0.736686,0.335071,0.897998
3,Probe,IsolationForest,2331,0.054421,0.951523,0.751610,0.839833,0.987545
5,Probe,OneClassSVM,2331,0.052714,0.869155,0.740497,0.799684,0.955750
4,Probe,LocalOutlierFactor,2331,0.052862,0.420849,0.579445,0.487575,0.893131
8,R2L,OneClassSVM,199,0.052714,0.251256,0.065789,0.104275,0.839396
7,R2L,LocalOutlierFactor,199,0.052862,0.110553,0.029973,0.047160,0.530313
6,R2L,IsolationForest,199,0.054421,0.100503,0.026560,0.042017,0.845110
11,U2R,OneClassSVM,10,0.052714,0.700000,0.009763,0.019257,0.918479


### Best untuned detector per attack family

,attack_category,algorithm,support,false_positive_rate,detection_rate,precision,f1
0,DoS,IsolationForest,9186,0.054421,0.969301,0.923939,0.946077
1,Probe,IsolationForest,2331,0.054421,0.951523,0.751610,0.839833
2,R2L,OneClassSVM,199,0.052714,0.251256,0.065789,0.104275
3,U2R,OneClassSVM,10,0.052714,0.700000,0.009763,0.019257


## Specialist fine-tuning

For each family, the strongest base algorithm is tuned on the validation split. Candidate thresholds are derived from normal-training score quantiles; validation labels select the hyperparameters and threshold. The final metrics below remain from the untouched test split.

In [4]:
if 'best_base_by_attack' not in globals(): best_base_by_attack=pd.DataFrame({'attack_category':['DoS','Probe','R2L','U2R'],'algorithm':['IsolationForest','IsolationForest','OneClassSVM','OneClassSVM']})
PARAMETER_GRIDS={
 'IsolationForest':[{'n_estimators':200,'max_samples':0.7},{'n_estimators':300,'max_samples':1.0},{'n_estimators':500,'max_samples':0.7},{'n_estimators':500,'max_samples':1.0}],
 'LocalOutlierFactor':[{'n_neighbors':15},{'n_neighbors':25},{'n_neighbors':35},{'n_neighbors':50}],
 'OneClassSVM':[{'nu':0.01,'gamma':'scale'},{'nu':0.03,'gamma':'scale'},{'nu':0.05,'gamma':'scale'},{'nu':0.10,'gamma':'scale'},{'nu':0.05,'gamma':0.01}],
}
THRESHOLD_QUANTILES=[0.90,0.93,0.95,0.97,0.99]
tuned_specialists={}; tuned_results=[]; tuning_history=[]
for category in ATTACK_CATEGORIES:
    algorithm=str(best_base_by_attack.loc[best_base_by_attack.attack_category==category,'algorithm'].iloc[0])
    val_mask=(family_val=='Normal')|(family_val==category); Xv=X_val_all[val_mask]; yv=(family_val[val_mask]==category).astype(int)
    best=None
    for candidate in PARAMETER_GRIDS[algorithm]:
        model=build_detector(algorithm,candidate); model.fit(X_normal_train); train_scores=anomaly_score(model,X_normal_train); val_scores=anomaly_score(model,Xv)
        for quantile in THRESHOLD_QUANTILES:
            threshold=float(np.quantile(train_scores,quantile)); pred=(val_scores>=threshold).astype(int); m=binary_metrics(yv,pred,val_scores); record={'attack_category':category,'algorithm':algorithm,'params':candidate,'threshold_quantile':quantile,'threshold':threshold,**m}; tuning_history.append(record)
            key=(m['f1'],m['detection_rate'],-m['false_positive_rate'])
            if best is None or key>best['key']: best={'key':key,'model':model,'params':candidate,'threshold':threshold,'quantile':quantile,'val_scores':val_scores}
    model=best['model']; threshold=best['threshold']
    calibrator=LogisticRegression(class_weight='balanced',random_state=RANDOM_STATE).fit(best['val_scores'].reshape(-1,1),yv)
    test_mask=(family_test=='Normal')|(family_test==category); Xt=X_test_all[test_mask]; yt=(family_test[test_mask]==category).astype(int); test_scores=anomaly_score(model,Xt); pred=(test_scores>=threshold).astype(int); metrics=binary_metrics(yt,pred,test_scores)
    tuned_specialists[category]={'algorithm':algorithm,'model':model,'calibrator':calibrator,'threshold':threshold,'params':best['params']}
    run_name=f'Original_Normalvs{category}_{algorithm}_tuned'
    category_indices=test_idx[test_mask]; category_dataset=tracking_dataset(category_indices,'attack_category',f'normal_vs_{category}_tuned_test_original')
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({'task':'tuned_attack_specialist','data_variant':'Original','attack_category':category,'algorithm':algorithm,'validation_labels_used_for_tuning':'true'})
        mlflow.log_input(normal_training_dataset,context='normal_only_training'); mlflow.log_input(category_dataset,context='held_out_attack_wise_evaluation')
        mlflow.log_params({**DEFAULT_PARAMS[algorithm],**best['params'],'algorithm':algorithm,'attack_category':category,'selected_threshold':threshold,'selected_threshold_quantile':best['quantile'],'parameter_candidates':len(PARAMETER_GRIDS[algorithm]),'threshold_candidates':len(THRESHOLD_QUANTILES),'normal_train_rows':len(normal_train_idx),'validation_rows':len(yv),'test_rows':len(yt),'encoded_dimensions':X_normal_train.shape[1]})
        mlflow.log_metrics({k:float(v) for k,v in metrics.items()})
        mlflow.log_metrics({'calibration_coefficient':float(calibrator.coef_[0,0]),'calibration_intercept':float(calibrator.intercept_[0])})
        fig=binary_figure(yt,pred,test_scores,threshold,run_name); mlflow.log_figure(fig,f'plots/{category}_tuned_diagnostics.png'); plt.close(fig)
        family_history=[r for r in tuning_history if r['attack_category']==category]; mlflow.log_dict(family_history,'tuning/validation_search.json'); mlflow.log_dict(common_metadata,'metadata/run_metadata.json'); mlflow.sklearn.log_model(sk_model=model,name='tuned_anomaly_detector'); mlflow.sklearn.log_model(sk_model=calibrator,name='score_calibrator')
        tuned_results.append({'run_name':run_name,'run_id':run.info.run_id,'attack_category':category,'algorithm':algorithm,'threshold':threshold,'support':int(yt.sum()),**metrics})
tuned_results_df=pd.DataFrame(tuned_results).sort_values('detection_rate',ascending=False).reset_index(drop=True)
display(tuned_results_df[['attack_category','algorithm','support','false_positive_rate','detection_rate','precision','recall','f1','roc_auc']])
fig,ax=plt.subplots(figsize=(8,4)); ordered=tuned_results_df.sort_values('detection_rate',ascending=False); ax.bar(ordered.attack_category,ordered.detection_rate,color='steelblue'); ax.set_ylim(0,1.05); ax.set_ylabel('Detection rate'); ax.set_title('Tuned specialist detection rate by attack family'); ax.grid(axis='y',alpha=.25); plt.show()

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


,attack_category,algorithm,support,false_positive_rate,detection_rate,precision,recall,f1,roc_auc
0,DoS,IsolationForest,9186,0.008315,0.939582,0.987190,0.939582,0.962798,0.991340
1,Probe,IsolationForest,2331,0.031628,0.923638,0.834820,0.923638,0.876986,0.987892
2,U2R,OneClassSVM,10,0.008687,0.500000,0.040984,0.500000,0.075758,0.895397
3,R2L,OneClassSVM,199,0.009875,0.125628,0.158228,0.125628,0.140056,0.857870


C:\Users\Akhila\AppData\Local\Temp\ipykernel_29264\625108797.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(8,4)); ordered=tuned_results_df.sort_values('detection_rate',ascending=False); ax.bar(ordered.attack_category,ordered.detection_rate,color='steelblue'); ax.set_ylim(0,1.05); ax.set_ylabel('Detection rate'); ax.set_title('Tuned specialist detection rate by attack family'); ax.grid(axis='y',alpha=.25); plt.show()


## Multiclass anomaly-specialist probability ensemble

The four tuned specialist anomaly scores become inputs to a stacking classifier. A stratified portion of the validation split selects the stacker's regularization and class weighting; it is then refitted on the complete validation split. Its `predict_proba` output provides mutually exclusive Normal, DoS, Probe, R2L, and U2R probabilities.

In [5]:
def specialist_score_matrix(values): return np.column_stack([anomaly_score(tuned_specialists[c]['model'],values) for c in ATTACK_CATEGORIES])
validation_scores=specialist_score_matrix(X_val_all)
meta_train_idx,meta_select_idx=train_test_split(np.arange(len(family_val)),test_size=0.35,stratify=family_val,random_state=RANDOM_STATE+7)
stacking_search=[]; best_meta=None
for class_weight in [None,'balanced']:
    for C in [0.01,0.1,1.0,10.0,100.0]:
        candidate=Pipeline([('scale',StandardScaler()),('classifier',LogisticRegression(max_iter=3000,C=C,class_weight=class_weight,random_state=RANDOM_STATE))]); candidate.fit(validation_scores[meta_train_idx],family_val[meta_train_idx]); candidate_prediction=candidate.predict(validation_scores[meta_select_idx]); select_true=family_val[meta_select_idx]
        record={'class_weight':str(class_weight),'C':C,'balanced_accuracy':balanced_accuracy_score(select_true,candidate_prediction),'macro_f1':f1_score(select_true,candidate_prediction,average='macro',zero_division=0),'accuracy':accuracy_score(select_true,candidate_prediction)}; stacking_search.append(record); key=(record['balanced_accuracy'],record['macro_f1'],record['accuracy'])
        if best_meta is None or key>best_meta['key']: best_meta={'key':key,'class_weight':class_weight,'C':C}
meta_model=Pipeline([('scale',StandardScaler()),('classifier',LogisticRegression(max_iter=3000,C=best_meta['C'],class_weight=best_meta['class_weight'],random_state=RANDOM_STATE))]).fit(validation_scores,family_val)
class AttackProbabilityEnsemble:
    def __init__(self,preprocessor,specialists,categories,meta_model): self.preprocessor=preprocessor; self.specialists=specialists; self.categories=list(categories); self.meta_model=meta_model; self.classes_=np.array(['Normal']+list(categories))
    def predict_proba(self,frame):
        values=self.preprocessor.transform(frame[list(ORIGINAL_FEATURES)]); scores=np.column_stack([-np.asarray(self.specialists[c]['model'].decision_function(values)).ravel() for c in self.categories]); raw=self.meta_model.predict_proba(scores); positions=[list(self.meta_model.classes_).index(c) for c in self.classes_]; return raw[:,positions]
    def predict(self,frame): return self.classes_[np.argmax(self.predict_proba(frame),axis=1)]
ensemble=AttackProbabilityEnsemble(preprocessor,tuned_specialists,ATTACK_CATEGORIES,meta_model)
test_frame=X.iloc[test_idx].copy(); probabilities=ensemble.predict_proba(test_frame); predicted=ensemble.predict(test_frame); class_order=ensemble.classes_.tolist()
true_positions=np.array([class_order.index(label) for label in family_test]); manual_log_loss=float(-np.mean(np.log(np.clip(probabilities[np.arange(len(probabilities)),true_positions],1e-15,1))))
ensemble_metrics={'accuracy':accuracy_score(family_test,predicted),'balanced_accuracy':balanced_accuracy_score(family_test,predicted),'macro_f1':f1_score(family_test,predicted,average='macro',zero_division=0),'weighted_f1':f1_score(family_test,predicted,average='weighted',zero_division=0),'multiclass_log_loss':manual_log_loss,'macro_ovr_roc_auc':roc_auc_score(label_binarize(family_test,classes=class_order),probabilities,average='macro',multi_class='ovr')}
p,r,f,s=precision_recall_fscore_support(family_test,predicted,labels=class_order,zero_division=0); per_class_df=pd.DataFrame({'class':class_order,'support':s,'precision':p,'detection_rate_recall':r,'f1':f})
conf=confusion_matrix(family_test,predicted,labels=class_order)
fig,axes=plt.subplots(1,2,figsize=(13,5)); ConfusionMatrixDisplay(confusion_matrix=conf,display_labels=class_order).plot(cmap='Blues',ax=axes[0],colorbar=False); axes[0].set_title('Ensemble confusion matrix'); axes[1].bar(per_class_df['class'],per_class_df['detection_rate_recall']); axes[1].set_ylim(0,1.05); axes[1].set_ylabel('Detection rate / recall'); axes[1].set_title('Detection rate by class'); axes[1].tick_params(axis='x',rotation=30); fig.tight_layout()
input_example=test_frame.head(5).copy(); probability_columns=[f'probability_{c}' for c in class_order]; output_example=pd.DataFrame(ensemble.predict_proba(input_example),columns=probability_columns)
class ProbabilityPyfunc(mlflow.pyfunc.PythonModel):
    def __init__(self,model): self.model=model
    def predict(self,context,model_input,params=None): return pd.DataFrame(self.model.predict_proba(model_input),columns=[f'probability_{c}' for c in self.model.classes_])
ensemble_dataset=tracking_dataset(test_idx,'attack_category','multiclass_ensemble_test_original')
with mlflow.start_run(run_name='Original_AttackProbabilityEnsemble') as run:
    mlflow.set_tags({'task':'multiclass_anomaly_probability_ensemble','data_variant':'Original','output_type':'normalized_class_probabilities','classes':','.join(class_order)})
    mlflow.log_input(normal_training_dataset,context='specialist_normal_training'); mlflow.log_input(ensemble_dataset,context='held_out_multiclass_evaluation')
    mlflow.log_params({'specialist_count':len(tuned_specialists),'original_feature_count':len(ORIGINAL_FEATURES),'encoded_dimensions':X_normal_train.shape[1],'normal_train_rows':len(normal_train_idx),'test_rows':len(test_idx),'ensemble_method':'logistic_regression_stacking','meta_class_weight':str(best_meta['class_weight']),'meta_C':best_meta['C'],'meta_selection_objective':'validation_balanced_accuracy_then_macro_f1',**{f'{c}_algorithm':tuned_specialists[c]['algorithm'] for c in ATTACK_CATEGORIES}})
    mlflow.log_metrics({k:float(v) for k,v in ensemble_metrics.items()}); mlflow.log_figure(fig,'plots/ensemble_confusion_and_detection_rates.png')
    mlflow.log_table(per_class_df,'metrics/per_class_metrics.json'); mlflow.log_table(pd.DataFrame(classification_report(family_test,predicted,labels=class_order,output_dict=True,zero_division=0)).T.reset_index(names='class'),'metrics/classification_report.json')
    mlflow.log_table(input_example.assign(expected_attack_category=family_test[:len(input_example)]),'contracts/input_example.json'); mlflow.log_table(output_example,'contracts/output_probability_example.json')
    mlflow.log_dict({'input':{'type':'pandas.DataFrame','required_columns':list(ORIGINAL_FEATURES),'dtypes':common_metadata['input_schema']},'output':{'type':'pandas.DataFrame','columns':probability_columns,'constraints':['each value is in [0,1]','each row sums to 1']},'predicted_label_rule':'argmax probability'},'contracts/input_output_schema.json')
    mlflow.log_dict({c:{'algorithm':v['algorithm'],'parameters':v['params'],'threshold':v['threshold']} for c,v in tuned_specialists.items()},'metadata/specialist_manifest.json'); mlflow.log_table(pd.DataFrame(stacking_search),'tuning/stacking_validation_search.json')
    signature=infer_signature(input_example,output_example); mlflow.pyfunc.log_model(name='attack_probability_ensemble',python_model=ProbabilityPyfunc(ensemble),signature=signature,input_example=input_example)
    ensemble_run_id=run.info.run_id
plt.show(); display(pd.DataFrame([ensemble_metrics])); display(per_class_df); display(Markdown('### Example probability output')); display(output_example)

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/05 13:06:56 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute

2026/08/05 13:06:56 INFO mlflow.pyfunc: Validating input example against model signature


C:\Users\Akhila\AppData\Local\Temp\ipykernel_29264\338612502.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); display(pd.DataFrame([ensemble_metrics])); display(per_class_df); display(Markdown('### Example probability output')); display(output_example)


,accuracy,balanced_accuracy,macro_f1,weighted_f1,multiclass_log_loss,macro_ovr_roc_auc
0,0.781147,0.729655,0.482043,0.827191,0.770138,0.930457


,class,support,precision,detection_rate_recall,f1
0,Normal,13469,0.983398,0.804811,0.885187
1,DoS,9186,0.928609,0.760396,0.836126
2,Probe,2331,0.401512,0.729301,0.517898
3,R2L,199,0.085082,0.753769,0.152905
4,U2R,10,0.009188,0.600000,0.018100


### Example probability output

,probability_Normal,probability_DoS,probability_Probe,probability_R2L,probability_U2R
0,9.300991e-03,0.213722,0.595056,0.137957,0.043963
1,5.755739e-08,0.556764,0.443042,0.000169,0.000025
2,8.450935e-01,0.000017,0.000030,0.078295,0.076564
3,8.449600e-01,0.000021,0.000036,0.079702,0.075281
4,3.146638e-08,0.463725,0.536102,0.000149,0.000024


In [6]:
easiest=tuned_results_df.sort_values(['detection_rate','f1'],ascending=False).iloc[0]; hardest=tuned_results_df.sort_values(['detection_rate','f1'],ascending=True).iloc[0]
if 'global_results_df' not in globals():
    experiment=mlflow.get_experiment_by_name(EXPERIMENT_NAME); tracked=mlflow.search_runs([experiment.experiment_id],filter_string="tags.task = 'global_anomaly_detection'").sort_values('start_time',ascending=False).drop_duplicates('tags.algorithm')
    global_results_df=tracked[['tags.algorithm','metrics.f1','metrics.detection_rate','metrics.false_positive_rate','metrics.roc_auc','metrics.average_precision']].rename(columns={'tags.algorithm':'algorithm','metrics.f1':'f1','metrics.detection_rate':'detection_rate','metrics.false_positive_rate':'false_positive_rate','metrics.roc_auc':'roc_auc','metrics.average_precision':'average_precision'}).sort_values('f1',ascending=False).reset_index(drop=True)
best_global=global_results_df.iloc[0]
specialist_lines='\n'.join([f"- **{r.attack_category}:** {r.algorithm}, detection rate {r.detection_rate:.3f}, FPR {r.false_positive_rate:.3f}, F1 {r.f1:.3f} (test attacks={int(r.support)})." for _,r in tuned_results_df.sort_values('attack_category').iterrows()])
display(Markdown(f'''# Conclusions

## Global anomaly detection
The best global original-feature detector by test F1 is **{best_global.algorithm}**, with F1 **{best_global.f1:.3f}**, detection rate **{best_global.detection_rate:.3f}**, false-positive rate **{best_global.false_positive_rate:.3f}**, ROC-AUC **{best_global.roc_auc:.3f}**, and average precision **{best_global.average_precision:.3f}**.

## Which attacks are easiest and hardest?
By held-out detection rate after validation tuning, **{easiest.attack_category}** is easiest ({easiest.detection_rate:.3f}) and **{hardest.attack_category}** is hardest ({hardest.detection_rate:.3f}). Interpret the rare U2R result cautiously because its test support is only {int(tuned_results_df.loc[tuned_results_df.attack_category=='U2R','support'].iloc[0])}. Detection rate must be read with FPR: a detector that flags everything would have high detection but poor operational value.

## Which algorithms specialize in each attack family?
{specialist_lines}

## Ensemble conclusion
The stacking probability ensemble achieves multiclass accuracy **{ensemble_metrics['accuracy']:.3f}**, balanced accuracy **{ensemble_metrics['balanced_accuracy']:.3f}**, macro-F1 **{ensemble_metrics['macro_f1']:.3f}**, and weighted-F1 **{ensemble_metrics['weighted_f1']:.3f}**. Its MLflow model accepts the 41 original columns and returns five normalized probability columns for Normal, DoS, Probe, R2L, and U2R. This is a hybrid semi-supervised system: detector fitting uses only normal traffic, while validation labels select specialist configurations and train the score-stacking classifier.

## Limitations
R2L and especially U2R have very small samples, so their estimates have high variance. Probability outputs are produced by a validation-trained stacker and are not guaranteed to match future traffic prevalence; recalibration on newer traffic is recommended before production use.''' ))

# Conclusions

## Global anomaly detection
The best global original-feature detector by test F1 is **IsolationForest**, with F1 **0.944**, detection rate **0.950**, false-positive rate **0.054**, ROC-AUC **0.988**, and average precision **0.986**.

## Which attacks are easiest and hardest?
By held-out detection rate after validation tuning, **DoS** is easiest (0.940) and **R2L** is hardest (0.126). Interpret the rare U2R result cautiously because its test support is only 10. Detection rate must be read with FPR: a detector that flags everything would have high detection but poor operational value.

## Which algorithms specialize in each attack family?
- **DoS:** IsolationForest, detection rate 0.940, FPR 0.008, F1 0.963 (test attacks=9186).
- **Probe:** IsolationForest, detection rate 0.924, FPR 0.032, F1 0.877 (test attacks=2331).
- **R2L:** OneClassSVM, detection rate 0.126, FPR 0.010, F1 0.140 (test attacks=199).
- **U2R:** OneClassSVM, detection rate 0.500, FPR 0.009, F1 0.076 (test attacks=10).

## Ensemble conclusion
The stacking probability ensemble achieves multiclass accuracy **0.781**, balanced accuracy **0.730**, macro-F1 **0.482**, and weighted-F1 **0.827**. Its MLflow model accepts the 41 original columns and returns five normalized probability columns for Normal, DoS, Probe, R2L, and U2R. This is a hybrid semi-supervised system: detector fitting uses only normal traffic, while validation labels select specialist configurations and train the score-stacking classifier.

## Limitations
R2L and especially U2R have very small samples, so their estimates have high variance. Probability outputs are produced by a validation-trained stacker and are not guaranteed to match future traffic prevalence; recalibration on newer traffic is recommended before production use.

# Final comparison and recommendation for a two-stage IDS

## Supervised multiclass versus stacked anomaly multiclass

| Held-out metric | Supervised XGBoost (`Experiment.ipynb`) | Stacked anomaly ensemble |
|---|---:|---:|
| Accuracy | **0.9993** | 0.7811 |
| Balanced accuracy | **0.9723** | 0.7297 |
| Macro-F1 | **0.9301** | 0.4820 |
| Weighted-F1 | **0.9993** | 0.8272 |
| Macro one-vs-rest ROC-AUC | **0.9999** | 0.9305 |
| Multiclass log loss | **0.0029** | 0.7701 |

The supervised model is the clear choice for assigning known attack families. The stacked anomaly ensemble detects some rare attacks, but its family predictions are not precise enough for the primary classifier: R2L precision is 0.0851 and U2R precision is 0.0092. Decision Tree has a marginally higher supervised macro-F1 (0.9324), but XGBoost is preferred for a probabilistic cascade because it has much higher balanced accuracy (0.9723), substantially better log loss, and near-perfect macro ROC-AUC.

> These results are indicative rather than a perfectly paired statistical comparison because the two notebooks currently use different held-out splits. A final benchmark should evaluate both approaches on the same untouched or time-based test set.

## Recommended two-stage architecture

1. **Stage 1 — Normal versus Attack: use supervised XGBoost.** Its binary test F1 is 0.9993, detection rate is 0.9991, and FPR is 0.00045. Isolation Forest reaches a 0.9505 detection rate but has a much higher 0.0544 FPR, so it should not be the primary gate. Tune the Stage-1 threshold toward high attack recall because every false negative prevents Stage 2 from seeing that event.
2. **Stage 2 — Attack-family classification: use supervised XGBoost retrained only on attack rows.** The current five-class XGBoost includes Normal during training. For the actual cascade, retrain and calibrate it on DoS, Probe, R2L, and U2R only, because Stage 2 receives traffic already classified as Attack. Use balanced weighting and report macro-F1 and per-family recall due to the severe R2L/U2R imbalance.
3. **Keep anomaly detection as a parallel safety branch.** Run Isolation Forest or the anomaly ensemble alongside Stage 1. If supervised XGBoost predicts Normal but the anomaly score is high, route the event to an `Unknown/Suspicious` queue instead of forcing it into a known family. This is where anomaly detection adds value for novel or distribution-shifted attacks.

### Final selection

Use **supervised binary XGBoost → attack-only supervised multiclass XGBoost** as the main two-stage pipeline. Use the anomaly model as an auxiliary unknown-attack detector and disagreement signal, not as the main multiclass classifier. Validate the complete cascade end to end, since its effective attack-family recall equals the probability of passing Stage 1 multiplied by Stage-2 recall.

# Stage 2 — Attack-Only Novelty Experiments

This section implements the two novelty components required after Stage 1 has routed a record as an attack:

- **Model 2 — Global attack novelty:** determine whether an attack resembles the overall population of known attack families.
- **Model 3 — Attack-category novelty:** after the supervised Random Forest predicts a family, determine whether the record resembles known examples of that predicted family.

All models use the original 41 features and the same three algorithms used earlier in this notebook. These experiments do not modify the supervised Stage-2 classifier.

## Stage-2 methodology and limitations

### Model 2 evaluation

For each algorithm, four leave-one-family-out folds are run. One family is removed from fitting and treated as pseudo-unknown on the external test split. The remaining families are known attacks. A 99th-percentile threshold from a separate known-attack reference split controls false novelty alerts.

### Model 3 evaluation

Each category model is fitted only on training examples from its own attack family. Test examples from that family are in-category records; all other attack families are out-of-category records. Selection balances category acceptance and other-family rejection.

This evaluates rejection of known-but-different families. It is a proxy for novelty, not proof of future zero-day detection. U2R results are especially uncertain because only 31 training and 10 test records are available.

In [7]:
# Stage-2 experiment configuration
from sklearn.base import BaseEstimator

S2_THRESHOLD_QUANTILE = 0.99
S2_GLOBAL_FIT_LIMIT = 6000
S2_GLOBAL_REFERENCE_LIMIT = 3000
S2_CATEGORY_FIT_LIMIT = 6000
S2_MODEL_VERSION = "stage2-attack-novelty-v1"

S2_ALGORITHM_SLUG = {
    "IsolationForest": "isolationforest",
    "LocalOutlierFactor": "localoutlierfactor",
    "OneClassSVM": "oneclasssvm",
}
S2_CATEGORY_RUN_LABEL = {
    "DoS": "Dos",
    "Probe": "Probe",
    "R2L": "R2L",
    "U2R": "U2R",
}

attack_train_idx = train_idx[
    families.iloc[train_idx].to_numpy() != "Normal"
]
attack_test_idx = test_idx[
    families.iloc[test_idx].to_numpy() != "Normal"
]

stage2_support = pd.DataFrame(
    {
        "train_attacks": families.iloc[attack_train_idx].value_counts(),
        "test_attacks": families.iloc[attack_test_idx].value_counts(),
    }
).fillna(0).astype(int).reindex(ATTACK_CATEGORIES)

display(stage2_support)
print("Global and category novelty threshold quantile:", S2_THRESHOLD_QUANTILE)


,train_attacks,test_attacks
attack_category,,
DoS,27556,9186
Probe,6994,2331
R2L,597,199
U2R,31,10


Global and category novelty threshold quantile: 0.99


## Shared attack-novelty utilities

In [8]:
def make_stage2_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numeric = [column for column in ORIGINAL_FEATURES if column not in categorical]
    return ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric,
            ),
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def balanced_index_sample(indices, limit, seed):
    indices = np.asarray(indices)
    if len(indices) <= limit:
        return indices
    index_families = families.iloc[indices].to_numpy()
    observed = sorted(np.unique(index_families).tolist())
    per_family = max(1, limit // len(observed))
    rng = np.random.default_rng(seed)
    selected = []
    for category in observed:
        candidates = indices[index_families == category]
        selected.extend(
            rng.choice(
                candidates,
                size=min(per_family, len(candidates)),
                replace=False,
            ).tolist()
        )
    remaining = limit - len(selected)
    if remaining > 0:
        unused = np.setdiff1d(indices, np.asarray(selected), assume_unique=False)
        if len(unused):
            selected.extend(
                rng.choice(
                    unused,
                    size=min(remaining, len(unused)),
                    replace=False,
                ).tolist()
            )
    return np.asarray(selected, dtype=int)


def stage2_detector(algorithm, fit_rows):
    params = dict(DEFAULT_PARAMS[algorithm])
    if algorithm == "LocalOutlierFactor":
        params["n_neighbors"] = min(
            int(params["n_neighbors"]), max(2, fit_rows - 1)
        )
    if algorithm == "IsolationForest":
        return IsolationForest(random_state=RANDOM_STATE, **params), params
    if algorithm == "LocalOutlierFactor":
        return LocalOutlierFactor(**params), params
    return OneClassSVM(**params), params


class OriginalAttackNoveltyModel(BaseEstimator):
    "Raw-original-feature novelty model where larger scores mean more novel."

    def __init__(
        self,
        preprocessor,
        detector,
        reference_scores,
        threshold,
        scope,
        algorithm,
        model_version=S2_MODEL_VERSION,
    ):
        self.preprocessor = preprocessor
        self.detector = detector
        self.reference_scores = np.sort(np.asarray(reference_scores, dtype=float))
        self.threshold = float(threshold)
        self.scope = scope
        self.algorithm = algorithm
        self.model_version = model_version
        self.required_features = list(ORIGINAL_FEATURES)

    def decision_function(self, frame):
        transformed = self.preprocessor.transform(frame[self.required_features])
        return -np.asarray(self.detector.decision_function(transformed)).ravel()

    def novelty_percentile(self, frame):
        scores = self.decision_function(frame)
        ranks = np.searchsorted(self.reference_scores, scores, side="right")
        return (ranks + 1.0) / (len(self.reference_scores) + 1.0)

    def predict(self, frame):
        return (self.decision_function(frame) >= self.threshold).astype(int)

    def predict_details(self, frame):
        scores = self.decision_function(frame)
        return pd.DataFrame(
            {
                "anomaly_score": scores,
                "novelty_percentile": self.novelty_percentile(frame),
                "is_novel": (scores >= self.threshold).astype(int),
                "scope": self.scope,
                "algorithm": self.algorithm,
                "model_version": self.model_version,
            },
            index=frame.index,
        )


def fit_stage2_novelty_model(fit_indices, reference_indices, algorithm, scope):
    fit_frame = X.iloc[fit_indices][list(ORIGINAL_FEATURES)].copy()
    reference_frame = X.iloc[reference_indices][list(ORIGINAL_FEATURES)].copy()
    transformer = make_stage2_preprocessor(fit_frame)
    fit_values = transformer.fit_transform(fit_frame)
    reference_values = transformer.transform(reference_frame)
    detector, effective_params = stage2_detector(algorithm, len(fit_frame))
    started = time.perf_counter()
    detector.fit(fit_values)
    fit_seconds = time.perf_counter() - started
    reference_scores = anomaly_score(detector, reference_values)
    threshold = float(np.quantile(reference_scores, S2_THRESHOLD_QUANTILE))
    model = OriginalAttackNoveltyModel(
        transformer,
        detector,
        reference_scores,
        threshold,
        scope,
        algorithm,
    )
    return model, effective_params, fit_seconds, fit_values.shape[1]


def stage2_novelty_metrics(y_true, prediction, scores):
    values = binary_metrics(y_true, prediction, scores)
    values["known_acceptance_rate"] = values["specificity"]
    values["novel_rejection_rate"] = values["detection_rate"]
    acceptance = values["known_acceptance_rate"]
    rejection = values["novel_rejection_rate"]
    values["acceptance_rejection_hmean"] = (
        2 * acceptance * rejection / max(acceptance + rejection, 1e-12)
    )
    return values


def stage2_novelty_figure(y_true, prediction, scores, threshold, title):
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        prediction,
        display_labels=["Known/in-category", "Novel/out-of-category"],
        cmap="Blues",
        colorbar=False,
        ax=axes[0, 0],
    )
    RocCurveDisplay.from_predictions(y_true, scores, ax=axes[0, 1])
    PrecisionRecallDisplay.from_predictions(y_true, scores, ax=axes[1, 0])
    axes[1, 1].hist(scores[y_true == 0], bins=35, alpha=0.6, label="Known/in-category")
    axes[1, 1].hist(scores[y_true == 1], bins=35, alpha=0.6, label="Novel/out-of-category")
    axes[1, 1].axvline(threshold, color="black", linestyle="--", label="Reference threshold")
    axes[1, 1].legend()
    axes[1, 1].set_title("Novelty-score distribution")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def stage2_tracking_dataset(indices, name):
    frame = X.iloc[indices][list(ORIGINAL_FEATURES)].copy()
    frame["attack_category"] = families.iloc[indices].to_numpy()
    return mlflow.data.from_pandas(
        frame,
        source=str(data_path.resolve()),
        targets="attack_category",
        name=name,
    )


def stage2_model_example(model, indices):
    example = X.iloc[np.asarray(indices)[:5]][list(ORIGINAL_FEATURES)].copy()
    return example, model.predict_details(example)


# Model 2 — Global Attack Novelty

One run is created per algorithm using the required format `Original_global_attack_{Algorithm}`. Each run contains four leave-one-family-out evaluations, a final model trained on all known attack categories, plots for every pseudo-unknown family, parameters, datasets, metrics, the raw-input model, and its input/output contract.

In [9]:
global_attack_fold_rows = []
global_attack_run_rows = []
global_attack_models = {}

for algorithm in ALGORITHMS:
    algorithm_fold_rows = []
    fold_figures = []
    total_fold_fit_seconds = 0.0

    for fold_number, held_out_family in enumerate(ATTACK_CATEGORIES):
        known_train = train_idx[
            (families.iloc[train_idx].to_numpy() != "Normal")
            & (families.iloc[train_idx].to_numpy() != held_out_family)
        ]
        known_labels = families.iloc[known_train].to_numpy()
        fold_fit, fold_reference = train_test_split(
            known_train,
            test_size=0.20,
            stratify=known_labels,
            random_state=RANDOM_STATE + 100 + fold_number,
        )
        fold_fit = balanced_index_sample(
            fold_fit,
            S2_GLOBAL_FIT_LIMIT,
            RANDOM_STATE + 200 + fold_number,
        )
        fold_reference = balanced_index_sample(
            fold_reference,
            S2_GLOBAL_REFERENCE_LIMIT,
            RANDOM_STATE + 300 + fold_number,
        )

        fold_model, effective_params, fold_fit_seconds, encoded_dimensions = (
            fit_stage2_novelty_model(
                fold_fit,
                fold_reference,
                algorithm,
                scope=f"global_known_attacks_excluding_{held_out_family}",
            )
        )
        total_fold_fit_seconds += fold_fit_seconds

        evaluation_indices = attack_test_idx
        evaluation_frame = X.iloc[evaluation_indices][list(ORIGINAL_FEATURES)]
        evaluation_family = families.iloc[evaluation_indices].to_numpy()
        y_pseudo_unknown = (evaluation_family == held_out_family).astype(int)
        scores = fold_model.decision_function(evaluation_frame)
        prediction = (scores >= fold_model.threshold).astype(int)
        fold_metrics = stage2_novelty_metrics(
            y_pseudo_unknown,
            prediction,
            scores,
        )

        fold_row = {
            "algorithm": algorithm,
            "held_out_pseudo_unknown_family": held_out_family,
            "known_fit_rows": len(fold_fit),
            "known_reference_rows": len(fold_reference),
            "known_test_rows": int((y_pseudo_unknown == 0).sum()),
            "pseudo_unknown_test_rows": int((y_pseudo_unknown == 1).sum()),
            "threshold": fold_model.threshold,
            "encoded_dimensions": encoded_dimensions,
            "fit_seconds": fold_fit_seconds,
            **fold_metrics,
        }
        algorithm_fold_rows.append(fold_row)
        global_attack_fold_rows.append(fold_row)

        figure = stage2_novelty_figure(
            y_pseudo_unknown,
            prediction,
            scores,
            fold_model.threshold,
            f"Global attack novelty — {algorithm} — pseudo-unknown {held_out_family}",
        )
        fold_figures.append((held_out_family, figure))

    fold_table = pd.DataFrame(algorithm_fold_rows)

    # Fit the deployable global model on every known attack family. Reference rows
    # remain separate so its novelty threshold and percentiles are leakage-safe.
    all_attack_labels = families.iloc[attack_train_idx].to_numpy()
    final_fit, final_reference = train_test_split(
        attack_train_idx,
        test_size=0.20,
        stratify=all_attack_labels,
        random_state=RANDOM_STATE + 400,
    )
    final_fit = balanced_index_sample(
        final_fit,
        S2_GLOBAL_FIT_LIMIT,
        RANDOM_STATE + 401,
    )
    final_reference = balanced_index_sample(
        final_reference,
        S2_GLOBAL_REFERENCE_LIMIT,
        RANDOM_STATE + 402,
    )
    final_model, final_params, final_fit_seconds, final_encoded_dimensions = (
        fit_stage2_novelty_model(
            final_fit,
            final_reference,
            algorithm,
            scope="global_known_attacks",
        )
    )

    known_test_frame = X.iloc[attack_test_idx][list(ORIGINAL_FEATURES)]
    known_test_scores = final_model.decision_function(known_test_frame)
    known_test_false_novel_rate = float(
        np.mean(known_test_scores >= final_model.threshold)
    )

    aggregate_metrics = {
        "lofo_macro_balanced_accuracy": float(fold_table["balanced_accuracy"].mean()),
        "lofo_macro_f1": float(fold_table["f1"].mean()),
        "lofo_macro_roc_auc": float(fold_table["roc_auc"].mean()),
        "lofo_macro_average_precision": float(fold_table["average_precision"].mean()),
        "lofo_macro_unknown_recall": float(fold_table["novel_rejection_rate"].mean()),
        "lofo_worst_family_unknown_recall": float(fold_table["novel_rejection_rate"].min()),
        "lofo_macro_known_false_novel_rate": float(fold_table["false_positive_rate"].mean()),
        "lofo_macro_acceptance_rejection_hmean": float(
            fold_table["acceptance_rejection_hmean"].mean()
        ),
        "final_known_attack_false_novel_rate": known_test_false_novel_rate,
        "fold_fit_seconds": total_fold_fit_seconds,
        "final_fit_seconds": final_fit_seconds,
    }

    run_name = f"Original_global_attack_{S2_ALGORITHM_SLUG[algorithm]}"
    fit_dataset = stage2_tracking_dataset(
        final_fit,
        f"global_attack_novelty_{algorithm}_fit",
    )
    reference_dataset = stage2_tracking_dataset(
        final_reference,
        f"global_attack_novelty_{algorithm}_reference",
    )
    test_dataset = stage2_tracking_dataset(
        attack_test_idx,
        f"global_attack_novelty_{algorithm}_test",
    )

    input_example, output_example = stage2_model_example(final_model, attack_test_idx)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags(
            {
                "task": "stage2_global_attack_novelty",
                "data_variant": "Original",
                "algorithm": algorithm,
                "training_population": "known_attacks_only",
                "evaluation_design": "leave_one_attack_family_out",
                "unknown_ground_truth": "pseudo_unknown_family_holdout",
                "model_version": S2_MODEL_VERSION,
            }
        )
        mlflow.log_input(fit_dataset, context="known_attack_model_fit")
        mlflow.log_input(reference_dataset, context="known_attack_novelty_reference")
        mlflow.log_input(test_dataset, context="attack_only_external_test")
        mlflow.log_params(
            {
                **final_params,
                "algorithm": algorithm,
                "threshold_quantile": S2_THRESHOLD_QUANTILE,
                "final_score_threshold": final_model.threshold,
                "original_feature_count": len(ORIGINAL_FEATURES),
                "encoded_dimensions": final_encoded_dimensions,
                "global_fit_limit": S2_GLOBAL_FIT_LIMIT,
                "global_reference_limit": S2_GLOBAL_REFERENCE_LIMIT,
                "final_fit_rows": len(final_fit),
                "final_reference_rows": len(final_reference),
                "external_attack_test_rows": len(attack_test_idx),
                "lofo_folds": len(ATTACK_CATEGORIES),
                "random_state": RANDOM_STATE,
                "model_version": S2_MODEL_VERSION,
            }
        )
        mlflow.log_metrics(aggregate_metrics)
        mlflow.log_table(
            fold_table,
            "evaluation/leave_one_family_out_metrics.json",
        )
        mlflow.log_dict(
            {
                "original_features": list(ORIGINAL_FEATURES),
                "known_attack_categories": ATTACK_CATEGORIES,
                "labels_used_for_model_fit": "attack category used only for balancing and family holdout",
                "score_orientation": "larger means more novel relative to known attacks",
                "threshold_source": "separate known-attack reference split",
                "limitation": "held-out known families are proxies for unknown attacks",
            },
            "metadata/run_metadata.json",
        )
        mlflow.log_dict(
            {
                "input": {
                    "type": "pandas.DataFrame",
                    "required_columns": list(ORIGINAL_FEATURES),
                },
                "output": {
                    "is_novel": {"0": "resembles known attacks", "1": "global attack novelty candidate"},
                    "anomaly_score": "larger means more novel",
                    "novelty_percentile": "relative to known-attack reference scores",
                },
                "decision_rule": "anomaly_score >= final_score_threshold",
            },
            "metadata/input_output_contract.json",
        )
        mlflow.log_table(
            input_example.reset_index(drop=True),
            "examples/input_example.json",
        )
        mlflow.log_table(
            output_example.reset_index(drop=True),
            "examples/output_example.json",
        )
        for held_out_family, figure in fold_figures:
            mlflow.log_figure(
                figure,
                f"plots/lofo_{held_out_family}_diagnostics.png",
            )
            plt.close(figure)
        signature = infer_signature(input_example, final_model.predict(input_example))
        mlflow.sklearn.log_model(
            final_model,
            name="global_attack_novelty_model",
            signature=signature,
            input_example=input_example,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
        run_id = run.info.run_id

    global_attack_models[algorithm] = final_model
    global_attack_run_rows.append(
        {
            "run_name": run_name,
            "run_id": run_id,
            "algorithm": algorithm,
            "threshold": final_model.threshold,
            **aggregate_metrics,
        }
    )
    print(
        f"{run_name}: balanced accuracy={aggregate_metrics['lofo_macro_balanced_accuracy']:.4f}, "
        f"unknown recall={aggregate_metrics['lofo_macro_unknown_recall']:.4f}, "
        f"known false-novel rate={aggregate_metrics['lofo_macro_known_false_novel_rate']:.4f}"
    )

global_attack_fold_df = pd.DataFrame(global_attack_fold_rows)
global_attack_results_df = pd.DataFrame(global_attack_run_rows).sort_values(
    ["lofo_macro_acceptance_rejection_hmean", "lofo_macro_roc_auc"],
    ascending=False,
).reset_index(drop=True)

display(global_attack_results_df)
display(
    global_attack_fold_df.pivot(
        index="held_out_pseudo_unknown_family",
        columns="algorithm",
        values="novel_rejection_rate",
    )
)


Original_global_attack_isolationforest: balanced accuracy=0.7367, unknown recall=0.4810, known false-novel rate=0.0076


Original_global_attack_localoutlierfactor: balanced accuracy=0.7393, unknown recall=0.4870, known false-novel rate=0.0084


Original_global_attack_oneclasssvm: balanced accuracy=0.9187, unknown recall=0.8451, known false-novel rate=0.0076


,run_name,run_id,algorithm,threshold,lofo_macro_balanced_accuracy,lofo_macro_f1,lofo_macro_roc_auc,lofo_macro_average_precision,lofo_macro_unknown_recall,lofo_worst_family_unknown_recall,lofo_macro_known_false_novel_rate,lofo_macro_acceptance_rejection_hmean,final_known_attack_false_novel_rate,fold_fit_seconds,final_fit_seconds
0,Original_global_attack_oneclasssvm,1d3ecbadc26a456cba7bafdf793ca70b,OneClassSVM,2.793586,0.918747,0.693002,0.917797,0.791302,0.845102,0.800000,0.007609,0.911504,0.006055,0.392260,0.107699
1,Original_global_attack_isolationforest,0c49f2656da14142ac16a0cd82acb7d5,IsolationForest,-0.079574,0.736682,0.493691,0.969345,0.599113,0.480950,0.286432,0.007586,0.621460,0.008954,1.165705,0.286296
2,Original_global_attack_localoutlierfactor,db61f022dab7453da8546b306bbf2269,LocalOutlierFactor,2.694872,0.739265,0.432701,0.946550,0.670857,0.486951,0.184470,0.008420,0.621358,0.007419,1.747411,0.096719


algorithm,IsolationForest,LocalOutlierFactor,OneClassSVM
held_out_pseudo_unknown_family,,,
DoS,0.808840,0.776399,0.808513
Probe,0.528529,0.184470,0.812098
R2L,0.286432,0.386935,0.959799
U2R,0.300000,0.600000,0.800000


## Model 2 selection

In [10]:
best_global_attack = global_attack_results_df.iloc[0]

fig_global_summary, axes = plt.subplots(1, 2, figsize=(13, 4))
global_attack_results_df.plot(
    x="algorithm",
    y=["lofo_macro_unknown_recall", "lofo_macro_known_false_novel_rate"],
    kind="bar",
    ax=axes[0],
)
axes[0].set_title("Global attack novelty: recall and false novelty")
axes[0].set_ylabel("Rate")
axes[0].tick_params(axis="x", rotation=0)

global_attack_fold_df.pivot(
    index="held_out_pseudo_unknown_family",
    columns="algorithm",
    values="novel_rejection_rate",
).plot(kind="bar", ax=axes[1])
axes[1].set_title("Pseudo-unknown recall by held-out family")
axes[1].set_ylabel("Recall")
axes[1].tick_params(axis="x", rotation=0)
fig_global_summary.tight_layout()
display(fig_global_summary)

with mlflow.start_run(run_id=str(best_global_attack.run_id)):
    mlflow.log_figure(fig_global_summary, "comparison/global_attack_novelty_summary.png")
    mlflow.log_table(global_attack_results_df, "comparison/all_global_algorithms.json")
plt.close(fig_global_summary)

display(
    Markdown(
        f"""### Model 2 winner: {best_global_attack.algorithm}

The best global attack-novelty model by the mean acceptance/rejection harmonic score is **{best_global_attack.algorithm}**. Across leave-one-family-out simulations it achieves mean balanced accuracy **{best_global_attack.lofo_macro_balanced_accuracy:.4f}**, mean pseudo-unknown recall **{best_global_attack.lofo_macro_unknown_recall:.4f}**, worst-family recall **{best_global_attack.lofo_worst_family_unknown_recall:.4f}**, mean known-attack false-novel rate **{best_global_attack.lofo_macro_known_false_novel_rate:.4f}**, and macro ROC-AUC **{best_global_attack.lofo_macro_roc_auc:.4f}**.

This model is the best measured choice among the three tested algorithms, but the held-out families are pseudo-unknowns. Production approval still requires temporally newer or genuinely unseen attack families."""
    )
)


<Figure size 1300x400 with 2 Axes>

### Model 2 winner: OneClassSVM

The best global attack-novelty model by the mean acceptance/rejection harmonic score is **OneClassSVM**. Across leave-one-family-out simulations it achieves mean balanced accuracy **0.9187**, mean pseudo-unknown recall **0.8451**, worst-family recall **0.8000**, mean known-attack false-novel rate **0.0076**, and macro ROC-AUC **0.9178**.

This model is the best measured choice among the three tested algorithms, but the held-out families are pseudo-unknowns. Production approval still requires temporally newer or genuinely unseen attack families.

# Model 3 — Attack-Category Novelty

Twelve runs are created: four attack families × three algorithms. The required naming scheme is used, for example `Original_Dos_isolationforest`. A category model returns `is_novel=1` when a record does not resemble the selected category.

In [11]:
category_novelty_rows = []
category_novelty_models = {}

for category_number, category in enumerate(ATTACK_CATEGORIES):
    category_train = train_idx[
        families.iloc[train_idx].to_numpy() == category
    ]
    category_fit, category_reference = train_test_split(
        category_train,
        test_size=0.20,
        random_state=RANDOM_STATE + 500 + category_number,
    )
    if len(category_fit) > S2_CATEGORY_FIT_LIMIT:
        category_fit = np.random.default_rng(
            RANDOM_STATE + 600 + category_number
        ).choice(
            category_fit,
            size=S2_CATEGORY_FIT_LIMIT,
            replace=False,
        )

    evaluation_indices = attack_test_idx
    evaluation_family = families.iloc[evaluation_indices].to_numpy()
    y_out_of_category = (evaluation_family != category).astype(int)
    evaluation_frame = X.iloc[evaluation_indices][list(ORIGINAL_FEATURES)]

    for algorithm in ALGORITHMS:
        model, effective_params, fit_seconds, encoded_dimensions = (
            fit_stage2_novelty_model(
                category_fit,
                category_reference,
                algorithm,
                scope=f"known_{category}",
            )
        )
        scores = model.decision_function(evaluation_frame)
        prediction = (scores >= model.threshold).astype(int)
        metrics = stage2_novelty_metrics(
            y_out_of_category,
            prediction,
            scores,
        )
        metrics["fit_seconds"] = fit_seconds

        per_source_category = []
        for source_category in ATTACK_CATEGORIES:
            source_mask = evaluation_family == source_category
            per_source_category.append(
                {
                    "category_model": category,
                    "source_attack_category": source_category,
                    "support": int(source_mask.sum()),
                    "novel_rejection_rate": float(prediction[source_mask].mean()),
                    "mean_anomaly_score": float(scores[source_mask].mean()),
                }
            )
        per_source_category_df = pd.DataFrame(per_source_category)

        run_name = (
            f"Original_{S2_CATEGORY_RUN_LABEL[category]}_"
            f"{S2_ALGORITHM_SLUG[algorithm]}"
        )
        fit_dataset = stage2_tracking_dataset(
            category_fit,
            f"{category}_{algorithm}_novelty_fit",
        )
        reference_dataset = stage2_tracking_dataset(
            category_reference,
            f"{category}_{algorithm}_novelty_reference",
        )
        test_dataset = stage2_tracking_dataset(
            evaluation_indices,
            f"{category}_{algorithm}_category_novelty_test",
        )
        input_example, output_example = stage2_model_example(
            model,
            evaluation_indices,
        )
        figure = stage2_novelty_figure(
            y_out_of_category,
            prediction,
            scores,
            model.threshold,
            f"{category} category novelty — {algorithm}",
        )

        with mlflow.start_run(run_name=run_name) as run:
            mlflow.set_tags(
                {
                    "task": "stage2_attack_category_novelty",
                    "data_variant": "Original",
                    "attack_category": category,
                    "algorithm": algorithm,
                    "training_population": f"{category}_only",
                    "positive_evaluation_label": "other_attack_category",
                    "model_version": S2_MODEL_VERSION,
                }
            )
            mlflow.log_input(fit_dataset, context="category_model_fit")
            mlflow.log_input(reference_dataset, context="category_novelty_reference")
            mlflow.log_input(test_dataset, context="attack_only_external_test")
            mlflow.log_params(
                {
                    **effective_params,
                    "algorithm": algorithm,
                    "attack_category": category,
                    "threshold_quantile": S2_THRESHOLD_QUANTILE,
                    "score_threshold": model.threshold,
                    "original_feature_count": len(ORIGINAL_FEATURES),
                    "encoded_dimensions": encoded_dimensions,
                    "category_fit_limit": S2_CATEGORY_FIT_LIMIT,
                    "category_fit_rows": len(category_fit),
                    "category_reference_rows": len(category_reference),
                    "category_test_rows": int((y_out_of_category == 0).sum()),
                    "other_attack_test_rows": int((y_out_of_category == 1).sum()),
                    "random_state": RANDOM_STATE,
                    "model_version": S2_MODEL_VERSION,
                }
            )
            mlflow.log_metrics({key: float(value) for key, value in metrics.items()})
            mlflow.log_table(
                per_source_category_df,
                "evaluation/rejection_by_source_attack_category.json",
            )
            mlflow.log_dict(
                {
                    "original_features": list(ORIGINAL_FEATURES),
                    "category_inlier": category,
                    "out_of_category_classes": [
                        value for value in ATTACK_CATEGORIES if value != category
                    ],
                    "score_orientation": "larger means more novel relative to the category",
                    "threshold_source": f"separate {category} reference split",
                    "category_train_support_warning": len(category_train) < 100,
                },
                "metadata/run_metadata.json",
            )
            mlflow.log_dict(
                {
                    "input": {
                        "type": "pandas.DataFrame",
                        "required_columns": list(ORIGINAL_FEATURES),
                    },
                    "output": {
                        "is_novel": {
                            "0": f"resembles known {category}",
                            "1": f"novel relative to {category}",
                        },
                        "anomaly_score": "larger means more category-novel",
                        "novelty_percentile": f"relative to held-out {category} reference scores",
                    },
                    "decision_rule": "anomaly_score >= score_threshold",
                },
                "metadata/input_output_contract.json",
            )
            mlflow.log_table(
                input_example.reset_index(drop=True),
                "examples/input_example.json",
            )
            mlflow.log_table(
                output_example.reset_index(drop=True),
                "examples/output_example.json",
            )
            mlflow.log_figure(figure, "plots/category_novelty_diagnostics.png")
            plt.close(figure)
            signature = infer_signature(input_example, model.predict(input_example))
            mlflow.sklearn.log_model(
                model,
                name="attack_category_novelty_model",
                signature=signature,
                input_example=input_example,
                serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            )
            run_id = run.info.run_id

        row = {
            "run_name": run_name,
            "run_id": run_id,
            "attack_category": category,
            "algorithm": algorithm,
            "fit_rows": len(category_fit),
            "reference_rows": len(category_reference),
            "category_test_support": int((y_out_of_category == 0).sum()),
            "threshold": model.threshold,
            **metrics,
        }
        category_novelty_rows.append(row)
        category_novelty_models[(category, algorithm)] = model
        print(
            f"{run_name}: acceptance={metrics['known_acceptance_rate']:.4f}, "
            f"other-family rejection={metrics['novel_rejection_rate']:.4f}, "
            f"balanced accuracy={metrics['balanced_accuracy']:.4f}"
        )

category_novelty_results_df = pd.DataFrame(category_novelty_rows)
category_novelty_results_df = category_novelty_results_df.sort_values(
    ["attack_category", "acceptance_rejection_hmean", "roc_auc"],
    ascending=[True, False, False],
).reset_index(drop=True)

display(
    category_novelty_results_df[
        [
            "run_name",
            "attack_category",
            "algorithm",
            "fit_rows",
            "reference_rows",
            "category_test_support",
            "known_acceptance_rate",
            "novel_rejection_rate",
            "acceptance_rejection_hmean",
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "false_positive_rate",
        ]
    ]
)


Original_Dos_isolationforest: acceptance=0.9893, other-family rejection=0.5012, balanced accuracy=0.7453


Original_Dos_localoutlierfactor: acceptance=0.9905, other-family rejection=0.4886, balanced accuracy=0.7396


Original_Dos_oneclasssvm: acceptance=0.9901, other-family rejection=0.9094, balanced accuracy=0.9498


Original_Probe_isolationforest: acceptance=0.9910, other-family rejection=0.8244, balanced accuracy=0.9077


Original_Probe_localoutlierfactor: acceptance=0.9880, other-family rejection=0.8090, balanced accuracy=0.8985


...

Original_R2L_oneclasssvm: acceptance=0.9950, other-family rejection=0.8657, balanced accuracy=0.9303


Original_U2R_isolationforest: acceptance=1.0000, other-family rejection=0.8310, balanced accuracy=0.9155


Original_U2R_localoutlierfactor: acceptance=0.9000, other-family rejection=0.8851, balanced accuracy=0.8926


Original_U2R_oneclasssvm: acceptance=1.0000, other-family rejection=0.8858, balanced accuracy=0.9429


,run_name,attack_category,algorithm,fit_rows,reference_rows,category_test_support,known_acceptance_rate,novel_rejection_rate,acceptance_rejection_hmean,balanced_accuracy,f1,roc_auc,false_positive_rate
0,Original_Dos_oneclasssvm,DoS,OneClassSVM,6000,5512,9186,0.990094,0.909449,0.948059,0.949771,0.935033,0.995055,0.009906
1,Original_Dos_isolationforest,DoS,IsolationForest,6000,5512,9186,0.989332,0.501181,0.665320,0.745256,0.650984,0.981289,0.010668
2,Original_Dos_localoutlierfactor,DoS,LocalOutlierFactor,6000,5512,9186,0.990529,0.488583,0.654386,0.739556,0.641675,0.975357,0.009471
3,Original_Probe_isolationforest,Probe,IsolationForest,5595,1399,2331,0.990991,0.824375,0.900037,0.907683,0.902628,0.975327,0.009009
4,Original_Probe_localoutlierfactor,Probe,LocalOutlierFactor,5595,1399,2331,0.987988,0.809047,0.889609,0.898518,0.892975,0.971469,0.012012
5,Original_Probe_oneclasssvm,Probe,OneClassSVM,5595,1399,2331,0.990991,0.095051,0.173463,0.543021,0.173247,0.957600,0.009009
6,Original_R2L_oneclasssvm,R2L,OneClassSVM,477,120,199,0.994975,0.865707,0.925850,0.930341,0.927977,0.995814,0.005025
7,Original_R2L_isolationforest,R2L,IsolationForest,477,120,199,0.989950,0.812093,0.892245,0.901022,0.896218,0.990796,0.010050
8,Original_R2L_localoutlierfactor,R2L,LocalOutlierFactor,477,120,199,1.000000,0.769411,0.869680,0.884705,0.869680,0.980937,0.000000
9,Original_U2R_oneclasssvm,U2R,OneClassSVM,24,7,10,1.000000,0.885797,0.939441,0.942899,0.939441,0.978354,0.000000


## Model 3 selection and final Stage-2 recommendation

In [12]:
best_category_models_df = (
    category_novelty_results_df.sort_values(
        ["acceptance_rejection_hmean", "roc_auc"],
        ascending=False,
    )
    .groupby("attack_category", as_index=False)
    .first()
    [
        [
            "attack_category",
            "algorithm",
            "run_name",
            "run_id",
            "fit_rows",
            "reference_rows",
            "category_test_support",
            "known_acceptance_rate",
            "novel_rejection_rate",
            "acceptance_rejection_hmean",
            "balanced_accuracy",
            "roc_auc",
        ]
    ]
)

display(Markdown("### Best category-novelty model for each attack family"))
display(best_category_models_df)

fig_category_summary, axes = plt.subplots(1, 2, figsize=(14, 4))
category_novelty_results_df.pivot(
    index="attack_category",
    columns="algorithm",
    values="known_acceptance_rate",
).plot(kind="bar", ax=axes[0])
axes[0].set_title("Acceptance of the claimed attack category")
axes[0].set_ylabel("Acceptance rate")
axes[0].tick_params(axis="x", rotation=0)

category_novelty_results_df.pivot(
    index="attack_category",
    columns="algorithm",
    values="novel_rejection_rate",
).plot(kind="bar", ax=axes[1])
axes[1].set_title("Rejection of other attack categories")
axes[1].set_ylabel("Rejection rate")
axes[1].tick_params(axis="x", rotation=0)
fig_category_summary.tight_layout()
display(fig_category_summary)

for row in best_category_models_df.itertuples(index=False):
    with mlflow.start_run(run_id=str(row.run_id)):
        mlflow.log_table(
            best_category_models_df,
            "comparison/best_category_novelty_models.json",
        )
        mlflow.log_figure(
            fig_category_summary,
            "comparison/category_novelty_summary.png",
        )
plt.close(fig_category_summary)

category_lines = []
for row in best_category_models_df.itertuples(index=False):
    reliability = (
        "LOW CONFIDENCE because support is very small"
        if row.fit_rows + row.reference_rows < 100
        else "supported by the available held-out split"
    )
    category_lines.append(
        f"- **{row.attack_category}: {row.algorithm}** — category acceptance "
        f"{row.known_acceptance_rate:.4f}, other-family rejection "
        f"{row.novel_rejection_rate:.4f}, balanced accuracy "
        f"{row.balanced_accuracy:.4f}, ROC-AUC {row.roc_auc:.4f}; {reliability}."
    )

display(
    Markdown(
        f"""# Stage-2 novelty model decision

## Model 2 — Global attack novelty

Select **{best_global_attack.algorithm}**, run `{best_global_attack.run_name}`. It provides the strongest measured balance between accepting known attack traffic and rejecting leave-one-family-out pseudo-unknown attacks. Its final logged model is fitted on all four known attack families using a separate known-attack reference split for the threshold.

## Model 3 — Category novelty

Use a category-specific model bank rather than one universal category detector:

{chr(10).join(category_lines)}

## Recommended Stage-2 sequence

1. The supervised attack-only Random Forest predicts `DoS`, `Probe`, `R2L`, or `U2R`.
2. The global **{best_global_attack.algorithm}** model checks whether the record resembles the overall known-attack population.
3. The novelty model corresponding to the Random Forest's predicted category checks whether the record resembles that category.
4. Accept the category only when classifier confidence is sufficient and both novelty checks agree. High global novelty indicates an unknown-attack candidate; low global novelty with high category novelty indicates likely family misclassification or a novel variant requiring review.

## Critical limitation

No true unknown-attack labels exist in this dataset. Leave-one-family-out evaluation is the strongest available proxy, but temporal or externally sourced unknown attacks are required before these novelty decisions can be considered production-validated. U2R requires additional data before its category model can be trusted."""
    )
)


### Best category-novelty model for each attack family

,attack_category,algorithm,run_name,run_id,fit_rows,reference_rows,category_test_support,known_acceptance_rate,novel_rejection_rate,acceptance_rejection_hmean,balanced_accuracy,roc_auc
0,DoS,OneClassSVM,Original_Dos_oneclasssvm,dcce917c4d0f4071b0e4437eb002da10,6000,5512,9186,0.990094,0.909449,0.948059,0.949771,0.995055
1,Probe,IsolationForest,Original_Probe_isolationforest,c5835846bfeb48108904a8cb03f6a776,5595,1399,2331,0.990991,0.824375,0.900037,0.907683,0.975327
2,R2L,OneClassSVM,Original_R2L_oneclasssvm,65b4fc481c54490aa94a8f43ab2a2d30,477,120,199,0.994975,0.865707,0.925850,0.930341,0.995814
3,U2R,OneClassSVM,Original_U2R_oneclasssvm,86f2813a1eaa433a9a96bcbea67ce4bc,24,7,10,1.000000,0.885797,0.939441,0.942899,0.978354


<Figure size 1400x400 with 2 Axes>

# Stage-2 novelty model decision

## Model 2 — Global attack novelty

Select **OneClassSVM**, run `Original_global_attack_oneclasssvm`. It provides the strongest measured balance between accepting known attack traffic and rejecting leave-one-family-out pseudo-unknown attacks. Its final logged model is fitted on all four known attack families using a separate known-attack reference split for the threshold.

## Model 3 — Category novelty

Use a category-specific model bank rather than one universal category detector:

- **DoS: OneClassSVM** — category acceptance 0.9901, other-family rejection 0.9094, balanced accuracy 0.9498, ROC-AUC 0.9951; supported by the available held-out split.
- **Probe: IsolationForest** — category acceptance 0.9910, other-family rejection 0.8244, balanced accuracy 0.9077, ROC-AUC 0.9753; supported by the available held-out split.
- **R2L: OneClassSVM** — category acceptance 0.9950, other-family rejection 0.8657, balanced accuracy 0.9303, ROC-AUC 0.9958; supported by the available held-out split.
- **U2R: OneClassSVM** — category acceptance 1.0000, other-family rejection 0.8858, balanced accuracy 0.9429, ROC-AUC 0.9784; LOW CONFIDENCE because support is very small.

## Recommended Stage-2 sequence

1. The supervised attack-only Random Forest predicts `DoS`, `Probe`, `R2L`, or `U2R`.
2. The global **OneClassSVM** model checks whether the record resembles the overall known-attack population.
3. The novelty model corresponding to the Random Forest's predicted category checks whether the record resembles that category.
4. Accept the category only when classifier confidence is sufficient and both novelty checks agree. High global novelty indicates an unknown-attack candidate; low global novelty with high category novelty indicates likely family misclassification or a novel variant requiring review.

## Critical limitation

No true unknown-attack labels exist in this dataset. Leave-one-family-out evaluation is the strongest available proxy, but temporal or externally sourced unknown attacks are required before these novelty decisions can be considered production-validated. U2R requires additional data before its category model can be trusted.